In [1]:
import pandas as pd

In [11]:
df = pd.read_csv("./data/dataset_report_anonimizzati_llama_3_1_8B_4bit.csv")
df = df.drop(columns="report_anonimo")
df

,nome_file,paese,periodo,inizio_periodo,fine_periodo,testo_originale
0,Gaza_Strip_Sep_2024_-_Apr_2025_KeyResults.txt,Gaza Strip,Sep 2024 / Apr 2025,Sep 2024,Apr 2025,"One year into the conflict, the risk of Famine..."
1,Afghanistan_Apr_2020_-_Nov_2020_KeyResults.txt,Afghanistan,Apr 2020 / Nov 2020,Apr 2020,Nov 2020,Food insecurity remains alarmingly high in Afg...
2,Afghanistan_Nov_2017_-_Feb_2018_KeyResults.txt,Afghanistan,Nov 2017 / Feb 2018,Nov 2017,Feb 2018,"During the 2017 post-harvest season, 33% of th..."
3,South_Sudan_Sep_2018_-_Mar_2019_KeyResults.txt,South Sudan,Sep 2018 / Mar 2019,Sep 2018,Mar 2019,"Based on the September IPC analysis, it is exp..."
4,Mozambique_Jun_2020_-_Sep_2020_KeyResults.txt,Mozambique,Jun 2020 / Sep 2020,Jun 2020,Sep 2020,The results of this Acute Food Insecurity pilo...
...,...,...,...,...,...,...
492,Haiti_Aug_2023_-_Jun_2024_KeyResults.txt,Haiti,Aug 2023 / Jun 2024,Aug 2023,Jun 2024,About 4.35 million people are experiencing hig...
493,Djibouti_Mar_2022_-_Dec_2022_KeyResults.txt,Djibouti,Mar 2022 / Dec 2022,Mar 2022,Dec 2022,For the current analysis period of March throu...
494,Djibouti_May_2013_-_May_2013_KeyResults.txt,Djibouti,May 2013 / May 2013,May 2013,May 2013,Food availability in the Republic of Djibouti ...
495,Madagascar_Sep_2024_-_Aug_2025_KeyResults.txt,Madagascar,Sep 2024 / Aug 2025,Sep 2024,Aug 2025,"Between September and December 2024, around 1...."


In [20]:
import re
from gliner import GLiNER

# 1. CARICAMENTO MODELLO
# Il modello multilingue gestisce bene sia report in inglese che in italiano
gliner_model = GLiNER.from_pretrained("urchade/gliner_large-v2")


# =====================================================================
# PUNTO 1: PREPROCESSING / MASKING PREVENTIVO
# =====================================================================
def preprocess_and_mask(text: str, model: GLiNER) -> str:
    """
    Applica una pulizia preliminare al testo prima di inviarlo all'LLM.
    - Sostituisce luoghi con termini generici
    - Maschera date e anni specifici
    - Maschera numeri di popolazione assoluti
    """
    # Define targets according to the three prompt rules
    labels = [
        "country", "region", "province", "district", "city", "village", "location", "landmark"
    ]

    # 1. Identificazione entità geografiche con GLiNER
    entities = model.predict_entities(text, labels, threshold=0.35)

    # Ordiniamo le entità da quella più lontana (offset maggiore) a quella iniziale.
    # Questo evita che le modifiche agli indici alterino le posizioni successive nel testo.
    entities_sorted = sorted(entities, key=lambda x: x['start'], reverse=True)

    masked_text = text
    for ent in entities_sorted:
        start = ent['start']
        end = ent['end']
        # Sostituzione con un segnaposto astratto / generico
        masked_text = masked_text[:start] + "[AFFECTED_AREA]" + masked_text[end:]

    # 2. Anonimizzazione delle date puntuali e anni tramite RegEx (Temporal Abstraction)
    # Rimuove anni specifici (es. 2024, 2025) mantenendo intatte le percentuali
    masked_text = re.sub(r'\b(19|20)\d{2}\b', '[YEAR]', masked_text)
    # Rimuove date composte (es. "4 December", "October 2024", "15/03/2024")
    masked_text = re.sub(
        r'\b\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{2,4}\b',
        '[DATE]',
        masked_text,
        flags=re.IGNORECASE
    )

    # 3. Anonimizzazione numeri assoluti di popolazione (Magnitude Abstraction)
    # Intercetta numeri grandi formattati con virgola/punto (es. 120,293 o 45.000) lasciando stare le percentuali (%)
    masked_text = re.sub(r'\b\d{1,3}(?:[,\.]\d{3})+\b(?!\s*%)', '[POPULATION_FIGURE]', masked_text)

    return masked_text


# =====================================================================
# PUNTO 2: POST-PROCESSING / VALIDAZIONE GUARDRAIL
# =====================================================================
def validate_llm_output(llm_output: str, model: GLiNER) -> dict:
    # 1. Etichette da cercare con GLiNER
    labels = ["country", "region", "province", "district", "city", "village", "location", "year", "specific date"]

    # Lista di termini generici ammessi dalle tue regole di Spatial Ablation
    ALLOWED_GENERIC_TERMS = [
        "affected area", "affected areas", "affected province",
        "province", "district", "coastal municipalities", "the province"
    ]

    gliner_matches = model.predict_entities(llm_output, labels, threshold=0.45)

    # Filtriamo le violazioni di GLiNER rimuovendo i termini generici ammessi
    gliner_violations = []
    for match in gliner_matches:
        if match['text'].lower() not in ALLOWED_GENERIC_TERMS:
            gliner_violations.append(match)

    # 2. Controllo RegEx corretto per gli anni (senza cattura parziale)
    regex_violations = []
    # Usiamo (?:19|20) come gruppo non-capturing per prendere l'intero anno
    year_matches = re.findall(r'\b(?:19|20)\d{2}\b', llm_output)
    if year_matches:
        regex_violations.append({
            "type": "Temporal Abstraction Violation",
            "detail": f"Anni a 4 cifre trovati: {year_matches}"
        })

    # Cerca numeri di popolazione con separatori (es. 120,293 o 45.000) ignorando le percentuali
    pop_matches = re.findall(r'\b\d{1,3}(?:[,\.]\d{3})+\b(?!\s*%)', llm_output)
    if pop_matches:
        regex_violations.append({
            "type": "Magnitude Abstraction Violation",
            "detail": f"Cifre assolute trovate: {pop_matches}"
        })

    is_valid = (len(gliner_violations) == 0) and (len(regex_violations) == 0)

    return {
        "is_valid": is_valid,
        "gliner_violations": gliner_violations,
        "regex_violations": regex_violations
    }


# =====================================================================
# ESEMPIO DI ESECUZIONE DELLA PIPELINE
# =====================================================================

raw_report = """
In October 2024, an estimated 120,293 people in the district of Upper Nile
are facing IPC Phase 4 acute food insecurity. During the post-harvest season,
approximately 45% of households in Malakal city will require assistance until mid-2025.
"""

print("--- 1. TEST PREPROCESSING ---")
preprocessed_text = preprocess_and_mask(raw_report, gliner_model)
print("Testo preprocessato prima dell'invio all'LLM:\n")
print(preprocessed_text)


print("\n--- 2. TEST VALIDAZIONE GUARDRAIL ---")
# Esempio A: Output LLM NON valido (ha dimenticato di rimuovere un luogo e un anno)
bad_llm_output = "In the affected area, 45% of households in Malakal require assistance until 2025."
result_bad = validate_llm_output(bad_llm_output, gliner_model)
print(f"Output A è valido? {result_bad['is_valid']}")
print(f"Violazioni rilevate: {result_bad['gliner_violations'] + result_bad['regex_violations']}\n")

# Esempio B: Output LLM Valido (Ablazione completa rispettata)
good_llm_output = "In the affected province, tens of thousands of individuals require assistance during the post-harvest season. Approximately 45% of households remain in IPC Phase 4."
result_good = validate_llm_output(good_llm_output, gliner_model)
print(f"Output B è valido? {result_good['is_valid']}")

/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

--- 1. TEST PREPROCESSING ---
Testo preprocessato prima dell'invio all'LLM:


In October [YEAR], an estimated [POPULATION_FIGURE] people in the district of [AFFECTED_AREA]
are facing IPC Phase 4 acute food insecurity. During the post-harvest season,
approximately 45% of households in [AFFECTED_AREA] will require assistance until mid-[YEAR].


--- 2. TEST VALIDAZIONE GUARDRAIL ---
Output A è valido? False
Violazioni rilevate: [{'start': 43, 'end': 50, 'text': 'Malakal', 'label': 'city', 'score': 0.8987544178962708}, {'start': 76, 'end': 80, 'text': '2025', 'label': 'year', 'score': 0.9947550296783447}, {'type': 'Temporal Abstraction Violation', 'detail': "Anni a 4 cifre trovati: ['2025']"}]

Output B è valido? True


In [12]:
import re
import pandas as pd
from gliner import GLiNER

# 1. CARICAMENTO MODELLO
gliner_model = GLiNER.from_pretrained("urchade/gliner_large-v2")


# =====================================================================
# FUNZIONE DI HELPER PER DIVIDERE IL TESTO IN FRASI (SENTENCE SPLITTING)
# =====================================================================
def split_into_sentences(text: str) -> list[str]:
    """
    Divide un testo lungo in singole frasi basandosi sulla punteggiatura finale.
    Usa un'espressione regolare che preserva i punti decimali e le sigle.
    """
    if not isinstance(text, str):
        return []

    # Divide per . ! ? seguiti da uno spazio o a capo, mantenendo puliti i segmenti
    sentence_endings = re.compile(r'(?<=[.!?])\s+')
    sentences = sentence_endings.split(text.strip())

    return [s for s in sentences if s]


# =====================================================================
# FUNZIONE DI ABLAZIONE PER SINGOLA FRASE
# =====================================================================
def anonymize_sentence(sentence: str, model: GLiNER) -> str:
    """
    Applica l'anonimizzazione con GLiNER e RegEx su una singola frase (lunghezza < 384 token).
    """
    spatial_labels = [
        "country", "region", "province", "district", "city",
        "village", "location", "landmark"
    ]

    # GLiNER analizza la frase (ora sicuramente sotto il limite di troncamento)
    entities = model.predict_entities(sentence, spatial_labels, threshold=0.35, max_len=512)

    # Sostituzione da destra verso sinistra per non alterare gli offset
    entities_sorted = sorted(entities, key=lambda x: x['start'], reverse=True)

    anonymized = sentence
    for ent in entities_sorted:
        start = ent['start']
        end = ent['end']
        anonymized = anonymized[:start] + "[AFFECTED_AREA]" + anonymized[end:]

    # Temporal Abstraction: Anni specifici a 4 cifre
    anonymized = re.sub(r'\b(?:19|20)\d{2}\b', '[DATE]', anonymized)

    # Temporal Abstraction: Date composte
    anonymized = re.sub(
        r'\b\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{2,4}\b',
        '[DATE]',
        anonymized,
        flags=re.IGNORECASE
    )

    # Magnitude Abstraction: Cifre assolute di popolazione (esclude %)
    anonymized = re.sub(
        r'\b\d{1,3}(?:[,\.]\d{3})+\b(?!\s*%)',
        '[POPULATION_FIGURE]',
        anonymized
    )

    return anonymized


# =====================================================================
# FUNZIONE PRINCIPALE: CHUNKING + ANONIMIZZAZIONE + RICOMPOSIZIONE
# =====================================================================
def anonymize_full_report(full_text: str, model: GLiNER) -> str:
    """
    Gestisce report di qualsiasi lunghezza:
    1. Spezza il testo in frasi.
    2. Processa ciascuna frase singolarmente con GLiNER (niente più UserWarning).
    3. Ricompone il testo finale.
    """
    if not isinstance(full_text, str) or not full_text.strip():
        return full_text

    # 1. Divisione in frasi
    sentences = split_into_sentences(full_text)

    # 2. Elaborazione frase per frase
    anonymized_sentences = [
        anonymize_sentence(s, model) for s in sentences
    ]

    # 3. Ricomposizione con lo spazio originale
    return " ".join(anonymized_sentences)


# =====================================================================
# INTEGRARLO NEL DATAFRAME PANDAS
# =====================================================================
def process_dataframe_safe(df: pd.DataFrame, input_col: str, output_col: str) -> pd.DataFrame:
    print(f"Elaborazione di {len(df)} report con gestione automatica della lunghezza frasi...")

    df[output_col] = df[input_col].apply(
        lambda x: anonymize_full_report(x, gliner_model)
    )

    return df


# =====================================================================
# ESEMPIO DI ESECUZIONE
# =====================================================================


/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [13]:
df_processed = process_dataframe_safe(df, input_col="testo_originale", output_col="gliner")
df_processed

Elaborazione di 497 report con gestione automatica della lunghezza frasi...


,nome_file,paese,periodo,inizio_periodo,fine_periodo,testo_originale,gliner
0,Gaza_Strip_Sep_2024_-_Apr_2025_KeyResults.txt,Gaza Strip,Sep 2024 / Apr 2025,Sep 2024,Apr 2025,"One year into the conflict, the risk of Famine...","One year into the conflict, the risk of Famine..."
1,Afghanistan_Apr_2020_-_Nov_2020_KeyResults.txt,Afghanistan,Apr 2020 / Nov 2020,Apr 2020,Nov 2020,Food insecurity remains alarmingly high in Afg...,Food insecurity remains alarmingly high in [AF...
2,Afghanistan_Nov_2017_-_Feb_2018_KeyResults.txt,Afghanistan,Nov 2017 / Feb 2018,Nov 2017,Feb 2018,"During the 2017 post-harvest season, 33% of th...","During the [DATE] post-harvest season, 33% of ..."
3,South_Sudan_Sep_2018_-_Mar_2019_KeyResults.txt,South Sudan,Sep 2018 / Mar 2019,Sep 2018,Mar 2019,"Based on the September IPC analysis, it is exp...","Based on the September IPC analysis, it is exp..."
4,Mozambique_Jun_2020_-_Sep_2020_KeyResults.txt,Mozambique,Jun 2020 / Sep 2020,Jun 2020,Sep 2020,The results of this Acute Food Insecurity pilo...,The results of this Acute Food Insecurity pilo...
...,...,...,...,...,...,...,...
492,Haiti_Aug_2023_-_Jun_2024_KeyResults.txt,Haiti,Aug 2023 / Jun 2024,Aug 2023,Jun 2024,About 4.35 million people are experiencing hig...,About 4.35 million people are experiencing hig...
493,Djibouti_Mar_2022_-_Dec_2022_KeyResults.txt,Djibouti,Mar 2022 / Dec 2022,Mar 2022,Dec 2022,For the current analysis period of March throu...,For the current analysis period of March throu...
494,Djibouti_May_2013_-_May_2013_KeyResults.txt,Djibouti,May 2013 / May 2013,May 2013,May 2013,Food availability in the Republic of Djibouti ...,Food availability in the [AFFECTED_AREA] is ma...
495,Madagascar_Sep_2024_-_Aug_2025_KeyResults.txt,Madagascar,Sep 2024 / Aug 2025,Sep 2024,Aug 2025,"Between September and December 2024, around 1....","Between September and December [DATE], around ..."


In [15]:
df_processed.to_csv("report_anonimizzati_gliner.csv")

In [16]:
df_processed.paese.unique()

array(['Gaza Strip', 'Afghanistan', 'South Sudan', 'Mozambique',
       'United Republic of Tanzania', 'El Salvador', 'Somalia',
       'Honduras', 'Sudan', 'Zimbabwe', 'Malawi', 'Pakistan', 'Djibouti',
       'Lebanon', 'Bangladesh', 'Kenya', 'Uganda', 'Dominican Republic',
       'Guatemala', 'Madagascar', 'Central African Republic',
       'Democratic Republic of the Congo', 'Yemen', 'Burundi', 'Eswatini',
       'Lesotho', 'Haiti', 'Angola', 'Ethiopia', 'Namibia', 'Tajikistan',
       'Zambia', 'Ecuador', 'Cambodia', 'South Africa', 'Timor-Leste'],
      dtype=object)

In [7]:
df_processed["gliner"][1]

'During the current period (March to May [DATE]), the [AFFECTED_AREA] of the [AFFECTED_AREA] is experiencing a moderate but sustained decline in acute food security, primarily driven by climatic variability, high food prices, and low food reserves. An estimated [POPULATION_FIGURE] people (16 percent of the analysed population) are classified in IPC Phase 3 or above (Crisis or worse), requiring urgent action to safeguard livelihoods and address food consumption gaps. This includes approximately [POPULATION_FIGURE] people in IPC Phase 3 (Crisis) and [POPULATION_FIGURE] in IPC Phase 4 (Emergency). The [AFFECTED_AREA] is the most severely affected, with 25 percent of its population in Phase 3 or higher, while [AFFECTED_AREA], [AFFECTED_AREA] and [AFFECTED_AREA] remain in Phase 2 (Stressed), despite hosting a significant population in Crisis. Food access and availability are constrained by crop losses caused by prolonged droughts, erratic rainfall, and pest outbreaks, and are further exacer

In [8]:
df_processed["testo_originale"][1]

'During the current period (March to May 2025), the Trinational Border Region of the Río Lempa is experiencing a moderate but sustained decline in acute food security, primarily driven by climatic variability, high food prices, and low food reserves. An estimated 100,000 people (16 percent of the analysed population) are classified in IPC Phase 3 or above (Crisis or worse), requiring urgent action to safeguard livelihoods and address food consumption gaps. This includes approximately 97,000 people in IPC Phase 3 (Crisis) and 3,000 in IPC Phase 4 (Emergency). The Ch’orti’ micro-region is the most severely affected, with 25 percent of its population in Phase 3 or higher, while Ocotepeque, Cayaguanca and Güija remain in Phase 2 (Stressed), despite hosting a significant population in Crisis. Food access and availability are constrained by crop losses caused by prolonged droughts, erratic rainfall, and pest outbreaks, and are further exacerbated by the increasing cost of basic food items.\n